# RAGAS 평가 기반 노트북

이 노트북은 Research Agent와 Final Answer Agent를 RAGAS로 평가하기 위한 기반 파일입니다.

현재 DB에 실제 RAG 데이터가 없을 수 있으므로, 이 노트북은 먼저 **평가 데이터셋 형식과 실행 흐름**을 이해하는 데 초점을 둡니다.

## 1. RAGAS 평가 구조

RAGAS 평가는 보통 아래 데이터를 사용합니다.

```text
question      사용자 질문
answer        모델이 생성한 최종 답변
contexts      RAG가 검색한 근거 문서들
ground_truth  사람이 작성한 기준 정답
```

Research Agent만 평가할 때는 `contexts` 품질이 중요합니다.
Final Answer까지 평가할 때는 `answer`가 context에 충실한지도 함께 봅니다.

In [ ]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "README.md").exists() and (path / "src").exists():
            return path
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")


project_root = find_project_root(Path.cwd())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

project_root

## 2. 평가 데이터셋 읽기

`evaluation_dataset_template.csv`는 아직 완성된 평가 데이터가 아니라 템플릿입니다.

처음에는 `question`과 `ground_truth`를 사람이 채우고, 나중에 Research Agent와 Final Answer Agent를 실행해서 `contexts`, `answer`를 채우면 됩니다.

In [ ]:
import pandas as pd

dataset_path = project_root / "src" / "agents" / "research" / "evaluation" / "evaluation_dataset_template.csv"
frame = pd.read_csv(dataset_path)
frame

## 3. 비어 있는 컬럼 확인

RAGAS를 실제로 실행하려면 `answer`와 `contexts`가 채워져야 합니다.

아래 셀은 현재 어떤 컬럼이 비어 있는지 확인합니다.

In [ ]:
required_columns = ["question", "answer", "contexts", "ground_truth"]

for column in required_columns:
    empty_count = frame[column].isna().sum() + (frame[column].fillna("").astype(str).str.strip() == "").sum()
    print(column, "empty_count=", int(empty_count))

## 4. Research Agent로 contexts 채우기 예시

아래 코드는 실제 DB가 준비된 뒤 사용할 예시입니다.

현재 DB에 데이터가 없다면 `retrieved_docs`가 비거나 `errors`에 실패 이유가 들어갈 수 있습니다.

In [ ]:
from src.agents.research import classify_research_route, run_research


def make_context_for_question(question: str) -> tuple[str, list[str]]:
    state = {
        "user_query": question,
        "character_name": "평가용캐릭터",
        "world_name": "스카니아",
    }
    route = classify_research_route(question)
    # 평가 준비 단계에서는 외부 Web API 의존성을 줄이기 위해 필요 시 Web을 끌 수 있습니다.
    # route["use_web"] = False
    result = run_research(state, route=route, auto_create_embedding=False)
    contexts = [doc.get("page_content", "") for doc in result.get("retrieved_docs", [])]
    return result.get("context", ""), contexts


# 예시 실행:
# context_text, contexts = make_context_for_question("노말 스우 요구 스펙 알려줘")
# print(context_text[:1000])

## 5. RAGAS 실행 예시

아래 셀은 `answer`와 `contexts`가 채워진 뒤 실행합니다.

주의할 점:

- `ragas`와 `datasets` 패키지가 설치되어 있어야 합니다.
- RAGAS 평가용 LLM/API key 설정이 필요할 수 있습니다.
- 현재 템플릿 데이터는 `answer`, `contexts`가 비어 있어서 바로 평가하면 안 됩니다.

In [ ]:
ready_frame = frame.dropna(subset=["answer", "contexts"])
ready_frame = ready_frame[
    (ready_frame["answer"].astype(str).str.strip() != "")
    & (ready_frame["contexts"].astype(str).str.strip() != "")
]

print("평가 가능 샘플 수:", len(ready_frame))

if len(ready_frame) == 0:
    print("아직 answer와 contexts가 채워진 샘플이 없습니다. 먼저 평가 데이터를 완성하세요.")

In [ ]:
# 실제 평가 데이터가 준비된 뒤 주석을 해제해서 사용하세요.
# from src.evaluation.ragas_eval import evaluate_rag_csv
#
# result = evaluate_rag_csv(
#     dataset_path,
#     output_path=project_root / "src" / "agents" / "research" / "evaluation" / "ragas_result.csv",
# )
# result

## 6. 체크리스트

RAGAS를 실제로 돌리기 전 체크리스트입니다.

- PostgreSQL에 `documents`, `document_chunks`, `document_embeddings`가 있다.
- Neo4j에 `Boss`, `Source`, `Reward` 같은 노드가 있다.
- Research Agent 실행 시 `retrieved_docs`가 1개 이상 나온다.
- Final Answer Agent가 `answer`를 만든다.
- 평가 CSV에 `question`, `answer`, `contexts`, `ground_truth`가 모두 채워져 있다.
- RAGAS 평가용 LLM/API key 설정이 되어 있다.